# Aula 12 - Notebook: Matriz de Incidência e Balanço de Massa Matricial

**Projeto:** SCADA-Core Automática — GRUPO 4  
**Unidade:** Planta Automatizada de Seleção e Classificação Óptica de Grãos com Separação Pneumática Tripla  
**Engenharia de Automação e Controle** — ECAA08  

---

### 1. Objetivos do Notebook
Neste notebook implementamos a **Matriz de Incidência Vértice-Aresta** $B \in \{-1, 0, 1\}^{n \times m}$ e resolvemos a equação matricial fundamental do balanço de conservação de massa em regime permanente:

$$B \cdot \vec{Q} = \vec{S}$$

Para o sistema de classificação de grãos, onde $\vec{Q} \in \mathbb{R}^m$ é o vetor de vazões em $\text{kg/h}$ (correlacionado com a variável calculada `FT-301` do CLP) e $\vec{S} \in \mathbb{R}^n$ é o vetor de balanço líquido nos equipamentos e silos coletores.

### 2. Funções de Formatação ASCII em Padrão Industrial

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII pura com suporte a infinitos."""
    larguras = [max(len(str(r)), 5) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "inf" if v == float('inf') else f"{v:.1f}" if isinstance(v, float) else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)


### 3. Modelo Estrutural da Planta de Grãos e Construtor da Rede Padrão

In [2]:
from typing import List, Dict, Any, Tuple

class GrafoTubulacao:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_tubulacao(self, origem: str, destino: str, comprimento_m: float,
                           tag_valvula: str, meio_transporte: str = "Duto", bitola: str = "DN200"):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Aresta": f"e{len(self.arestas_detalhes)+1}",
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Atuador / ISA": tag_valvula,
            "Meio": meio_transporte,
            "Bitola": bitola
        })

def criar_rede_padrao() -> GrafoTubulacao:
    nos = [
        "FN-101_Recepcao", "ALIM-101_Calha", "EST-201_Entrada", "BAL-301_Pesagem",
        "VIS-401_Inspecao", "DIV-501_Decisao", "EJE-602_EstacaoB", "EJE-603_EstacaoC",
        "CALHA-602_Secundario", "CALHA-603_Rejeito", "CALHA-701_Terminal",
        "SILO-701_Aprovados", "SILO-702_Secundario", "SILO-703_Rejeito"
    ]
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao("FN-101_Recepcao", "ALIM-101_Calha", 1.2, "XV-101", "Duto Vertical", "DN200")
    g.adicionar_tubulacao("ALIM-101_Calha", "EST-201_Entrada", 1.8, "c_ALIM", "Calha Vibratória", "L=300mm")
    g.adicionar_tubulacao("EST-201_Entrada", "BAL-301_Pesagem", 2.5, "c_EST/ST-201", "Correia", "L=400mm")
    g.adicionar_tubulacao("BAL-301_Pesagem", "VIS-401_Inspecao", 3.0, "WT-301/FT-301", "Correia", "L=400mm")
    g.adicionar_tubulacao("VIS-401_Inspecao", "DIV-501_Decisao", 1.0, "XS-401/KSA-401", "Zona Óptica", "L=400mm")
    g.adicionar_tubulacao("DIV-501_Decisao", "EJE-602_EstacaoB", 2.0, "ZC-602/p_POS602", "Correia", "L=400mm")
    g.adicionar_tubulacao("EJE-602_EstacaoB", "CALHA-602_Secundario", 0.8, "FY-602/ZSH-602", "Sopro B", "DN15")
    g.adicionar_tubulacao("EJE-602_EstacaoB", "EJE-603_EstacaoC", 1.5, "ST-201", "Correia", "L=400mm")
    g.adicionar_tubulacao("EJE-603_EstacaoC", "CALHA-603_Rejeito", 0.8, "FY-603/ZSH-601", "Sopro C", "DN15")
    g.adicionar_tubulacao("EJE-603_EstacaoC", "CALHA-701_Terminal", 2.2, "Descarga", "Correia", "L=400mm")
    g.adicionar_tubulacao("CALHA-701_Terminal", "SILO-701_Aprovados", 3.5, "XV-701", "Duto A", "DN250")
    g.adicionar_tubulacao("CALHA-602_Secundario", "SILO-702_Secundario", 4.0, "XV-702", "Duto B", "DN200")
    g.adicionar_tubulacao("CALHA-603_Rejeito", "SILO-703_Rejeito", 3.2, "XV-703", "Duto C", "DN200")
    g.adicionar_tubulacao("SILO-702_Secundario", "FN-101_Recepcao", 12.0, "XV-104", "Elevador Reciclo", "Canecas 6pol")
    return g

rede = criar_rede_padrao()


### 4. Classe `CalculadorIncidencia` e Geração da Matriz $B$

A classe estática `CalculadorIncidencia` constrói $B \in \{-1, 0, +1\}^{n \times m}$ a partir da lista de adjacências e detalhes de arestas e computa a Matriz Laplaciana $L = B B^T$.

In [3]:
class CalculadorIncidencia:
    @staticmethod
    def construir_matriz_incidencia(vertices: List[str], arestas: List[Dict[str, Any]]) -> Tuple[List[List[int]], List[str]]:
        n = len(vertices)
        m = len(arestas)
        v_idx = {v: i for i, v in enumerate(vertices)}
        B = [[0] * m for _ in range(n)]
        nomes_e = []
        for j, a in enumerate(arestas):
            u = v_idx[a["Origem"]]
            v = v_idx[a["Destino"]]
            B[u][j] = -1
            B[v][j] = 1
            nomes_e.append(f"e{j+1}:{a['Origem']}->{a['Destino']}")
        return B, nomes_e

    @staticmethod
    def calcular_laplaciano(B: List[List[int]]) -> List[List[int]]:
        n = len(B)
        m = len(B[0])
        L = [[0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                L[i][j] = sum(B[i][k] * B[j][k] for k in range(m))
        return L

B_mat, nomes_arestas = CalculadorIncidencia.construir_matriz_incidencia(rede.vertices, rede.arestas_detalhes)


### 5. Validação Estrutural e Resolução do Balanço de Massa ($S = B \cdot Q$)

In [4]:
print("=== MATRIZ DE INCIDÊNCIA VÉRTICE-ARESTA B (n x m) ===")
cols_rotulos = [f"e{j+1}" for j in range(len(rede.arestas_detalhes))]
print(formatar_matriz(B_mat, rede.vertices, cols_rotulos))

# 1. Validação estrita da soma por coluna nula
somas_col = [sum(B_mat[i][j] for i in range(len(rede.vertices))) for j in range(len(rede.arestas_detalhes))]
assert all(s == 0 for s in somas_col), "Erro Crítico: Coluna de incidência com soma não-nula!"
print("\n-> Validação Estrutural: Soma por coluna estritamente nula confirmada em todas as 14 arestas!")

# 2. Resolução do Balanço de Massa em Regime Permanente: S = B * Q (kg/h)
# Taxa nominal de alimentação = 1200 kg/h
# Cat B = 15% (180 kg/h), Cat C = 10% (120 kg/h), Cat A = 75% (900 kg/h)
# Reciclo e14 = 0.0 kg/h (em operação de expedição padrão)
Q_vec = [1200.0, 1200.0, 1200.0, 1200.0, 1200.0, 1200.0, 180.0, 1020.0, 120.0, 900.0, 900.0, 180.0, 120.0, 0.0]

S_res = []
for i, v_nome in enumerate(rede.vertices):
    balanco = sum(B_mat[i][j] * Q_vec[j] for j in range(len(Q_vec)))
    comportamento = "Fonte Externa (Injeção)" if balanco < -1e-5 else ("Sumidouro (Estocagem Silo)" if balanco > 1e-5 else "Regime Permanente (Conservado)")
    S_res.append({
        "Equipamento / Nó": v_nome,
        "Balanço Líquido S (kg/h)": f"{balanco:.1f}",
        "Estado Operacional": comportamento
    })

print("\n=== BALANÇO DE MASSA MATRICIAL EM REGIME PERMANENTE (S = B * Q) ===")
print(formatar_tabela(S_res))

balanco_global = sum(sum(B_mat[i][j] * Q_vec[j] for j in range(len(Q_vec))) for i in range(len(rede.vertices)))
print(f"\n-> Balanço Global da Planta: {balanco_global:.2f} kg/h (Conservação estrita da massa confirmada!)")


=== MATRIZ DE INCIDÊNCIA VÉRTICE-ARESTA B (n x m) ===
                     |    e1 |    e2 |    e3 |    e4 |    e5 |    e6 |    e7 |    e8 |    e9 |   e10 |   e11 |   e12 |   e13 |   e14
---------------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+------
FN-101_Recepcao      |    -1 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     1
ALIM-101_Calha       |     1 |    -1 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0
EST-201_Entrada      |     0 |     1 |    -1 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0
BAL-301_Pesagem      |     0 |     0 |     1 |    -1 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0
VIS-401_Inspecao     |     0 |     0 |     0 |     1 |    -1 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0
DIV-501_Decisao

### 6. Matriz Laplaciana $L = B B^T$ e Análise do Espaço de Ciclos de Kirchhoff

In [5]:
# Calculo da Matriz Laplaciana L = B * B^T
L_mat = CalculadorIncidencia.calcular_laplaciano(B_mat)
siglas = ["FN101", "ALIM", "EST201", "BAL301", "VIS401", "DIV501", "EJE602", "EJE603", "CAL602", "CAL603", "CAL701", "SIL701", "SIL702", "SIL703"]
print("=== MATRIZ LAPLACIANA DA PLANTA L = B * B^T (Topologia e Conectividade) ===")
print(formatar_matriz(L_mat, rede.vertices, siglas))

# Análise de Posto e Espaço de Ciclos
n = len(rede.vertices)
m = len(rede.arestas_detalhes)
rank_B = n - 1  # Grafo conexo unificado (c = 1)
dim_ciclos = m - rank_B

print("\n=== ANÁLISE TOPOLÓGICA DE POSTO E ESPAÇO DE CICLOS DE KIRCHHOFF ===")
print(f"-> Ordem do Grafo (|V| = n): {n}")
print(f"-> Tamanho do Grafo (|E| = m): {m}")
print(f"-> Posto da Matriz de Incidência rank(B): {rank_B}")
print(f"-> Dimensão do Espaço de Ciclos (Null Space dim(ker(B)) = m - n + 1): {dim_ciclos}")
print("-> Interpretação Industrial: O valor dim(ker(B)) = 1 reflete exatamente o circuito independente de reprocessamento e reciclo da Categoria B (e14).")


=== MATRIZ LAPLACIANA DA PLANTA L = B * B^T (Topologia e Conectividade) ===
                     | FN101 |  ALIM | EST201 | BAL301 | VIS401 | DIV501 | EJE602 | EJE603 | CAL602 | CAL603 | CAL701 | SIL701 | SIL702 | SIL703
---------------------+-------+-------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+-------
FN-101_Recepcao      |     2 |    -1 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |     -1 |      0
ALIM-101_Calha       |    -1 |     2 |     -1 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0
EST-201_Entrada      |     0 |    -1 |      2 |     -1 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0
BAL-301_Pesagem      |     0 |     0 |     -1 |      2 |     -1 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0
VIS-401_Inspecao     |     0 |     0 |      0 |     -1